### 14. 원본 기록을 1분씩 묶어 `cal01_base_v4` 만들기

- 같은 LOT 안에서 실제 관측이 있는 분만 묶음.
  - 강종이나 조업·정비 기록이 바뀌면 같은 분이어도 따로 묶음.
  - 관측이 없는 날은 채우지 않음.
- 구역별 후보 라벨과 기존 제외 구간은 유지함.
- 직전 60초를 보는 컬럼은 원본 기록에서 먼저 계산함.

### 1. 파일 이름과 집계 옵션

- `data/processed/`에 `cal01_base_v3.csv`를 넣음.
- `INPUT_FILE` 옵션에는 읽을 CSV 파일명을 적음.
- `OUTPUT_FILE` 옵션에는 만들 CSV 파일명을 적음.
- 전체 실행하면 v4, 원본 기록과의 연결표, 그림을 저장함.
- 조업이벤트·정비이력·수리계획 원본은 `data/` 또는 `data/raw/`에 함께 둠.
  - `data/raw/T-CR1-CAL01_온도.csv`가 있으면 `data/raw/`에서 읽고, 없으면 `data/`에서 읽음.
- 저장할 파일이 입력 파일과 같으면 멈춤


In [ ]:
INPUT_FILE = "cal01_base_v3.csv"  # 읽을 파일명: 원본 기록과 구역별 후보 라벨
OUTPUT_FILE = "cal01_base_v4.csv"  # 저장할 파일명: 같은 분을 묶은 데이터
SAVE_FILES = True  # True이면 CSV, 설정, 그림을 저장함
MAX_GAP_SECONDS = 15  # 원본 기록 사이가 15초를 넘으면 과거 계산을 다시 시작함
MIN_PAST_ROWS = 10  # 직전 60초의 유효 기록이 10개 이상일 때 평균과 표준편차를 계산함
MIN_MINUTE_ROWS = 10  # 1분 모델 입력에 필요한 최소 원본 기록 수
MIN_SPAN_SECONDS = 54  # 첫 관측부터 마지막 관측까지 1초를 더한 폭이 54초 미만이면 모델 입력을 보류함
NOMINAL_RAW_SECONDS = 1  # 기본 관측 간격인 1초를 마지막 관측 시각에 더해 시간폭을 계산함

import os
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager, rcParams
from IPython.display import display

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

ENC = "utf-8-sig"
DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
INPUT_CSV = os.path.join(PROCESSED_DIR, INPUT_FILE)
OUTPUT_CSV = os.path.join(PROCESSED_DIR, OUTPUT_FILE)

# 저장할 파일이 입력 원본과 같으면 덮어쓰지 않고 멈춤
if SAVE_FILES and os.path.normcase(os.path.realpath(INPUT_CSV)) == os.path.normcase(os.path.realpath(OUTPUT_CSV)):
    raise ValueError("입력 파일과 저장할 파일이 같음. 출력 파일명을 다르게 지정하세요.")

META_DIR = os.path.join(PROCESSED_DIR, "metadata", "v4")
REVIEW_DIR = os.path.join(PROCESSED_DIR, "review", "v4")
FIG_DIR = os.path.join(ROOT, "figures", "namelesspark", "14_minute_dataframe")
files = {
    "event": "G-02_조업이벤트.csv",
    "maintenance": "G-02_정비이력.csv",
    "repair_plan": "G-01_정기수리캘린더.csv",
}
for name, filename in files.items():
    files[name] = os.path.join(DATA_DIR, filename)

input_hashes = {}
hash_paths = {"base": INPUT_CSV}
hash_paths.update(files)
for name, path in hash_paths.items():
    with open(path, "rb") as source_file:
        input_hashes[name] = hashlib.sha256(source_file.read()).hexdigest()

font_names = []
for installed_font in font_manager.fontManager.ttflist:
    font_names.append(installed_font.name)
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font_name in font_names:
        rcParams["font.family"] = font_name
        break
rcParams["axes.unicode_minus"] = False

groups = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
    "Z9": ["TC-Z9"],
    "COOLING": ["TC-OUT1", "TC-OUT2"],
}
GRADE_GROUPS = ["Z3", "Z4", "Z5", "Z6", "Z7", "Z8", "Z9", "COOLING"]
SENSORS = []
LABEL_COLUMNS = []
for group, sensors in groups.items():
    SENSORS.extend(sensors)
    for suffix in ["LABEL", "LABEL_STATUS", "LABEL_REASON", "LABEL_SENSORS", "COMMON_REVIEW_IDS"]:
        LABEL_COLUMNS.append(group + "__" + suffix)

print("입력:", INPUT_CSV)
print("출력:", OUTPUT_CSV)


### 2. 조업·정비 기록을 연결하고 계산 구간 나누기

- 센서가 측정된 시각보다 앞선 강종변경 기록을 연결함.
- CAL01 정비 작업과 CR1 정기수리 계획은 따로 표시함.
- LOT, 15초 초과 공백, 제외 여부, 강종변경 기록이 바뀌면 과거 계산을 다시 시작함.
  - 정비 작업 또는 수리계획과 겹치는지가 바뀌어도 다시 시작함.
  - 작업번호나 제외 사유가 바뀌면 같은 분 안에서도 따로 집계함.
- 058의 정지와 061의 재가동은 팀 판단을 유지함.
- 정비 기록과 시간이 겹친다는 이유만으로 고장이나 정지로 정하지 않음.

In [ ]:
META = ["ROW_ID", "MEAS_DT", "LOT_NO", "GRADE", "EXCLUDED", "EXCLUSION_REASON", "DATA_KIND", "DATASET_VERSION"]
df = pd.read_csv(
    INPUT_CSV,
    encoding=ENC,
    usecols=META + SENSORS + LABEL_COLUMNS,
    dtype={"ROW_ID": "string", "LOT_NO": "string", "GRADE": "string"},
    low_memory=False,
)
df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"])
df["GRADE"] = df["GRADE"].fillna("미기록")
df["EXCLUDED"] = ~df["EXCLUDED"].astype("string").str.strip().str.lower().isin(["false", "0", "0.0"])
df["EXCLUSION_REASON"] = df["EXCLUSION_REASON"].fillna("")
for sensor in SENSORS:
    df[sensor] = pd.to_numeric(df[sensor], errors="coerce").replace([np.inf, -np.inf], np.nan)
for group in groups:
    df[group + "__LABEL"] = pd.to_numeric(df[group + "__LABEL"], errors="coerce")
    for suffix in ["LABEL_STATUS", "LABEL_REASON", "LABEL_SENSORS", "COMMON_REVIEW_IDS"]:
        column = group + "__" + suffix
        df[column] = df[column].fillna("").astype(str)

events = pd.read_csv(files["event"], encoding=ENC)
events = events.loc[events["PLANT_CD"].eq("CR1") & events["EVT_TYPE"].eq("강종변경")].copy()
events["EVT_DT"] = pd.to_datetime(events["EVT_DT"])
df = pd.merge_asof(
    df.sort_values("MEAS_DT"),
    events.sort_values("EVT_DT")[["EVT_ID", "EVT_DT", "BEF_VAL", "AFT_VAL"]],
    left_on="MEAS_DT",
    right_on="EVT_DT",
    direction="backward",
)
df = df.sort_values(["LOT_NO", "MEAS_DT"], kind="stable").reset_index(drop=True)
df["MINUTES_SINCE_GRADE_CHANGE"] = (df["MEAS_DT"] - df["EVT_DT"]).dt.total_seconds() / 60
df["EVENT_MATCH"] = df["GRADE"].eq(df["AFT_VAL"]).fillna(False) & df["EVT_DT"].notna()

maintenance = pd.read_csv(files["maintenance"], encoding=ENC)
maintenance = maintenance.loc[maintenance["EQP_TAG"].eq("P1-CR1-CAL01")].copy()
for column in ["STRT_DT", "END_DT"]:
    maintenance[column] = pd.to_datetime(maintenance[column])
df["MAINT_WORK_IDS"] = ""
for work in maintenance.itertuples(index=False):
    overlap = df["MEAS_DT"].ge(work.STRT_DT) & df["MEAS_DT"].lt(work.END_DT)
    df.loc[overlap, "MAINT_WORK_IDS"] += work.WO_NO + "|"
df["MAINT_WORK_IDS"] = df["MAINT_WORK_IDS"].str.rstrip("|")
df["MAINT_OVERLAP"] = df["MAINT_WORK_IDS"].ne("")

repair = pd.read_csv(files["repair_plan"], encoding=ENC)
repair = repair.loc[repair["PLANT_CD"].eq("CR1")].copy()
repair["STRT_DT"] = pd.to_datetime(repair["STRT_DT"])
repair["END_EXCLUSIVE"] = pd.to_datetime(repair["END_DT"]).dt.normalize() + pd.Timedelta(days=1)
df["REPAIR_PLAN_IDS"] = ""
for plan in repair.itertuples(index=False):
    overlap = df["MEAS_DT"].ge(plan.STRT_DT) & df["MEAS_DT"].lt(plan.END_EXCLUSIVE)
    df.loc[overlap, "REPAIR_PLAN_IDS"] += plan.PLAN_ID + "|"
df["REPAIR_PLAN_IDS"] = df["REPAIR_PLAN_IDS"].str.rstrip("|")

df["CONTEXT"] = "작업 기록 없음"
df.loc[df["MAINT_OVERLAP"], "CONTEXT"] = "정비 작업 기록과 겹침"
df.loc[df["REPAIR_PLAN_IDS"].ne(""), "CONTEXT"] = "정기수리 계획 기간"
df["OPERATING_NOTE"] = "별도 운전 상태 기록 없음"
df.loc[df["EXCLUDED"], "OPERATING_NOTE"] = df.loc[df["EXCLUDED"], "EXCLUSION_REASON"]
df.loc[df["EXCLUDED"] & df["LOT_NO"].eq("240058"), "OPERATING_NOTE"] = "정기수리를 위해 온도를 내림, 팀 확정"
df.loc[df["EXCLUDED"] & df["LOT_NO"].eq("240061"), "OPERATING_NOTE"] = "정기수리 후 재가동하며 온도를 올림, 팀 확정"

seconds = df["MEAS_DT"].diff().dt.total_seconds()
breaks = df["LOT_NO"].ne(df["LOT_NO"].shift()).fillna(True) | seconds.gt(MAX_GAP_SECONDS) | seconds.le(0)
for column in ["EXCLUDED", "GRADE", "EVT_ID", "CONTEXT"]:
    breaks = breaks | df[column].ne(df[column].shift()).fillna(True)
df["SEGMENT"] = breaks.cumsum()
df["BIN_START"] = df["MEAS_DT"].dt.floor("min")

# 작업번호나 제외 사유만 바뀐 경우에는 과거 60초 계산을 유지하고 분 집계만 나눔.
aggregation_keys = ["LOT_NO", "SEGMENT", "BIN_START", "EXCLUSION_REASON", "MAINT_WORK_IDS", "REPAIR_PLAN_IDS"]
df["MINUTE_ID"] = df.groupby(aggregation_keys, sort=False, dropna=False).ngroup()
print("원관측:", len(df), "/ 60초 계산 구간:", df["SEGMENT"].nunique(), "/ 1분 집계:", df["MINUTE_ID"].nunique())


### 3. 원본 기록에서 직전 60초 컬럼 계산하기

- 현재값, 초당 변화량, 직전 60초 평균과의 차이, 직전 60초 표준편차를 구함.
- 현재값은 과거 평균과 표준편차에 넣지 않음.
- 직전 60초 안에 유효한 기록이 10개 이상이면 계산함.
  - 60초 전체가 관측됐다는 뜻은 아님.
- 결측값과 제외 구간은 채우지 않음.

In [ ]:
features = {}
for sensor in SENSORS:
    feature = pd.DataFrame(np.nan, index=df.index, columns=["VALUE", "DIFF_PER_SEC", "DEV_60S", "STD_60S"])
    for segment, block in df.loc[~df["EXCLUDED"]].groupby("SEGMENT", sort=False):
        values = pd.Series(block[sensor].to_numpy(), index=pd.DatetimeIndex(block["MEAS_DT"]))
        elapsed = values.index.to_series().diff().dt.total_seconds()
        past = values.rolling("60s", closed="left", min_periods=MIN_PAST_ROWS)
        feature.loc[block.index, "VALUE"] = values.to_numpy()
        feature.loc[block.index, "DIFF_PER_SEC"] = values.diff().div(elapsed).to_numpy()
        feature.loc[block.index, "DEV_60S"] = values.sub(past.mean()).to_numpy()
        feature.loc[block.index, "STD_60S"] = past.std().to_numpy()
    features[sensor] = feature.replace([np.inf, -np.inf], np.nan).astype("float32")

group_inputs = {}
group_valid = {}
for group, sensors in groups.items():
    parts = []
    for sensor in sensors:
        parts.append(features[sensor].add_prefix(sensor + "__"))
    group_input = pd.concat(parts, axis=1)
    if group in GRADE_GROUPS:
        group_input[group + "__GRADE_CHANGE_LOG_MIN"] = np.log1p(df["MINUTES_SINCE_GRADE_CHANGE"])
    group_input = group_input.astype("float32")
    valid = np.isfinite(group_input.to_numpy()).all(axis=1) & ~df["EXCLUDED"].to_numpy()
    if group in GRADE_GROUPS:
        valid = valid & df["EVENT_MATCH"].to_numpy(dtype=bool)
    group_inputs[group] = group_input
    group_valid[group] = valid


### 4. 같은 분의 센서값과 모델 입력 묶기

- 각 센서는 평균·최솟값·최댓값·표준편차·유효 기록 수를 남김.
  - 평균은 기록된 값의 산술평균임.
  - 기록 사이의 시간 간격으로 가중치를 주지는 않음.
- 모델 입력은 같은 구역의 컬럼이 모두 계산된 원본 기록만 골라 평균 냄.
  - 입력이 부족한 기록이 하나라도 있으면 그 분의 해당 구역은 모델 판정을 보류함.
- 원본 기록이 10개 미만이거나 관측 시간폭이 54초 미만인 분도 저장함.
  - 해당 분은 모델 입력에서 보류함.
  - 두 값은 성능을 보기 전에 정한 입력 조건임.
  - 정상과 이상을 나누는 기준은 아님.
- 아래 표의 `집계 행`은 같은 분 안에서도 조업 조건이 달라 나뉜 행을 각각 센 개수임.
  - 한 분이 여러 행으로 나뉠 수 있어 행수를 그대로 분 단위 시간으로 읽지 않음.


In [ ]:
minute_groups = df.groupby("MINUTE_ID", sort=True)
minute = minute_groups.agg(
    LOT_NO=("LOT_NO", "first"),
    SEGMENT=("SEGMENT", "first"),
    BIN_START=("BIN_START", "first"),
    OBS_START=("MEAS_DT", "min"),
    OBS_END=("MEAS_DT", "max"),
    SOURCE_FIRST_ROW_ID=("ROW_ID", "first"),
    SOURCE_LAST_ROW_ID=("ROW_ID", "last"),
    ROW_COUNT=("ROW_ID", "size"),
    GRADE=("GRADE", "first"),
    EXCLUDED=("EXCLUDED", "first"),
    EXCLUSION_REASON=("EXCLUSION_REASON", "first"),
    EVT_ID=("EVT_ID", "first"),
    EVT_DT=("EVT_DT", "first"),
    BEF_VAL=("BEF_VAL", "first"),
    AFT_VAL=("AFT_VAL", "first"),
    EVENT_MATCH=("EVENT_MATCH", "all"),
    MINUTES_SINCE_GRADE_CHANGE=("MINUTES_SINCE_GRADE_CHANGE", "mean"),
    MAINT_OVERLAP=("MAINT_OVERLAP", "first"),
    MAINT_WORK_IDS=("MAINT_WORK_IDS", "first"),
    REPAIR_PLAN_IDS=("REPAIR_PLAN_IDS", "first"),
    CONTEXT=("CONTEXT", "first"),
    OPERATING_NOTE=("OPERATING_NOTE", "first"),
)
minute.insert(0, "ROW_ID", "M" + pd.Series(np.arange(1, len(minute) + 1), index=minute.index).astype(str).str.zfill(6))
minute["BIN_END"] = minute["BIN_START"] + pd.Timedelta(minutes=1)
minute["AVAILABLE_AT"] = minute["BIN_END"]
minute["MEAS_DT"] = minute["BIN_START"]
minute["OBSERVED_SPAN_SECONDS"] = (minute["OBS_END"] - minute["OBS_START"]).dt.total_seconds() + NOMINAL_RAW_SECONDS
minute["PARTIAL_MINUTE"] = minute["OBSERVED_SPAN_SECONDS"].lt(MIN_SPAN_SECONDS)
minute["DATA_KIND"] = "minute"
minute["DATASET_VERSION"] = "v4"
minute["FEATURE_PROFILE"] = "raw60s_groupmean_1min"

sensor_tables = []
for sensor in SENSORS:
    summary = minute_groups[sensor].agg(["mean", "min", "max", "std", "count"])
    summary = summary.rename(columns={
        "mean": sensor,
        "min": sensor + "__MIN",
        "max": sensor + "__MAX",
        "std": sensor + "__STD",
        "count": sensor + "__COUNT",
    })
    sensor_tables.append(summary)
minute = pd.concat([minute] + sensor_tables, axis=1)

input_columns_by_group = {}
for group, group_input in group_inputs.items():
    valid = pd.Series(group_valid[group], index=df.index)
    available_features = group_input.copy()
    available_features.loc[~valid] = np.nan
    average_features = available_features.groupby(df["MINUTE_ID"], sort=True).mean()
    minute = pd.concat([minute, average_features], axis=1)
    input_columns_by_group[group] = group_input.columns.tolist()
    minute[group + "__FEATURE_COUNT"] = valid.groupby(df["MINUTE_ID"]).sum().astype(int)
    enough = (
        ~minute["EXCLUDED"]
        & minute["ROW_COUNT"].ge(MIN_MINUTE_ROWS)
        & ~minute["PARTIAL_MINUTE"]
        & minute[group + "__FEATURE_COUNT"].eq(minute["ROW_COUNT"])
    )
    minute[group + "__MODEL_ELIGIBLE"] = enough
    minute[group + "__MODEL_STATUS"] = "모델 입력 가능"
    minute.loc[~enough, group + "__MODEL_STATUS"] = "구역 입력이 일부 원행에서 부족함"
    minute.loc[minute["PARTIAL_MINUTE"], group + "__MODEL_STATUS"] = "관측 시간폭 " + str(MIN_SPAN_SECONDS) + "초 미만"
    minute.loc[minute["ROW_COUNT"].lt(MIN_MINUTE_ROWS), group + "__MODEL_STATUS"] = "원관측 " + str(MIN_MINUTE_ROWS) + "개 미만"
    minute.loc[minute["EXCLUDED"], group + "__MODEL_STATUS"] = "기존 제외"

group_quality = []
for group in groups:
    group_quality.append({
        "구역": group,
        "집계 분": len(minute),
        "모델 입력 가능": int(minute[group + "__MODEL_ELIGIBLE"].sum()),
        "모델 입력 보류": int((~minute[group + "__MODEL_ELIGIBLE"]).sum()),
    })
display(pd.DataFrame(group_quality).rename(columns={"집계 분": "집계 행"}))


### 5. 짧게 나타난 후보도 구역 라벨에 남기기

- 그 분에 해당 구역의 라벨 1이 하나라도 있으면 1로 남김.
  - 분 전체가 계속 이상했다는 뜻은 아님.
- 해당 구역의 원본 라벨이 모두 0일 때만 0으로 남김.
- 나머지는 보류하고 기존 제외 분의 라벨은 비움.
- 라벨과 모델 입력 가능 여부는 따로 남김.
  - 라벨 1이어도 관측이 부족하면 모델 판정을 보류함.
- 아래 표의 `라벨 없음(보류·기존 제외)`는 판단을 보류한 행과 이미 제외한 행을 합친 개수임.


In [ ]:
label_summary_rows = []
for group in groups:
    minute = minute.copy()
    label_column = group + "__LABEL"
    one = df[label_column].eq(1)
    zero = df[label_column].eq(0)
    unknown = ~(one | zero)
    minute[group + "__LABEL_1_COUNT"] = one.groupby(df["MINUTE_ID"]).sum().astype(int)
    minute[group + "__LABEL_0_COUNT"] = zero.groupby(df["MINUTE_ID"]).sum().astype(int)
    minute[group + "__LABEL_UNKNOWN_COUNT"] = unknown.groupby(df["MINUTE_ID"]).sum().astype(int)
    minute[group + "__LABEL_1_RATIO"] = minute[group + "__LABEL_1_COUNT"] / minute["ROW_COUNT"]
    minute[group + "__LABEL_FIRST_1_DT"] = df["MEAS_DT"].where(one).groupby(df["MINUTE_ID"]).min()
    minute[group + "__LABEL_LAST_1_DT"] = df["MEAS_DT"].where(one).groupby(df["MINUTE_ID"]).max()
    minute[label_column] = np.nan
    positive = ~minute["EXCLUDED"] & minute[group + "__LABEL_1_COUNT"].gt(0)
    all_zero = ~minute["EXCLUDED"] & minute[group + "__LABEL_0_COUNT"].eq(minute["ROW_COUNT"])
    minute.loc[positive, label_column] = 1
    minute.loc[all_zero, label_column] = 0
    minute[group + "__LABEL_STATUS"] = "원자료 라벨에 보류가 있음"
    minute.loc[positive, group + "__LABEL_STATUS"] = "이 분에 구역 후보가 있었음"
    minute.loc[all_zero, group + "__LABEL_STATUS"] = "원자료 구역 라벨이 모두 0"
    minute.loc[minute["EXCLUDED"], group + "__LABEL_STATUS"] = "기존 제외"
    for suffix in ["LABEL_REASON", "LABEL_SENSORS", "COMMON_REVIEW_IDS"]:
        column = group + "__" + suffix
        minute[column] = ""
        selected = df
        if suffix in ["LABEL_REASON", "LABEL_SENSORS"]:
            positive_minutes = minute.index[positive]
            selected = df.loc[~df["MINUTE_ID"].isin(positive_minutes) | one]
        source_texts = selected.groupby("MINUTE_ID")[column].unique()
        for minute_id, values in source_texts.items():
            texts = []
            for value in values:
                if value:
                    texts.append(value)
            minute.loc[minute_id, column] = " | ".join(texts)
    label_summary_rows.append({
        "구역": group,
        "원관측 라벨 1": int(one.sum()),
        "1분 라벨 1": int(positive.sum()),
        "1분 라벨 0": int(all_zero.sum()),
        "1분 라벨 보류": int(minute[label_column].isna().sum()),
        "라벨 1이지만 모델 입력 보류": int((positive & ~minute[group + "__MODEL_ELIGIBLE"]).sum()),
    })

label_summary = pd.DataFrame(label_summary_rows)
display(label_summary.rename(columns={"1분 라벨 보류": "라벨 없음(보류·기존 제외)"}))


### 6. 069의 CP 급등을 1분 평균과 함께 보기

- 파란 선은 원본 기록임.
- 주황 선은 1분 평균임.
- 연한 주황색은 같은 분의 최솟값부터 최댓값까지임.

In [ ]:
plot_start = pd.Timestamp("2024-07-08 02:28:00")
plot_end = pd.Timestamp("2024-07-08 02:42:00")
raw_case = df.loc[df["LOT_NO"].eq("240069") & df["MEAS_DT"].ge(plot_start) & df["MEAS_DT"].lt(plot_end)]
minute_case = minute.loc[minute["LOT_NO"].eq("240069") & minute["BIN_START"].ge(plot_start) & minute["BIN_START"].lt(plot_end)]
display(pd.DataFrame([{
    "LOT": "240069",
    "비교 구간": "07/08 02:28~02:42",
    "원관측 CP 최고": raw_case["CP-Z4"].max(),
    "1분 평균 CP 최고": minute_case["CP-Z4"].max(),
    "분 최댓값에 남은 CP 최고": minute_case["CP-Z4__MAX"].max(),
    "구역 Z4 후보 분": int(minute_case["Z4__LABEL"].eq(1).sum()),
}]))

fig, axis = plt.subplots(figsize=(12, 4))
for segment, block in raw_case.groupby("SEGMENT", sort=False):
    axis.plot(block["MEAS_DT"], block["CP-Z4"], color="#396788", linewidth=0.8)
for segment, block in minute_case.groupby("SEGMENT", sort=False):
    if block.empty:
        continue
    edges = pd.concat([block["BIN_START"], pd.Series([block["BIN_END"].iloc[-1]])])
    average = np.r_[block["CP-Z4"].to_numpy(), block["CP-Z4"].iloc[-1]]
    low = np.r_[block["CP-Z4__MIN"].to_numpy(), block["CP-Z4__MIN"].iloc[-1]]
    high = np.r_[block["CP-Z4__MAX"].to_numpy(), block["CP-Z4__MAX"].iloc[-1]]
    axis.fill_between(edges, low, high, step="post", color="#edaa55", alpha=0.25)
    axis.step(edges, average, where="post", color="#d67b16", linewidth=1.5)
axis.plot([], [], color="#396788", label="원관측")
axis.plot([], [], color="#d67b16", label="1분 평균")
axis.fill_between([], [], [], color="#edaa55", alpha=0.25, label="분 최솟값~최댓값")
axis.set_title("069 CP 급등 | 평균과 최댓값을 함께 보존")
axis.set_ylabel("CP-Z4 기록값")
axis.set_xlabel("2024년 실제 시각")
axis.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
axis.legend()
axis.grid(alpha=0.15)
fig.tight_layout()
if SAVE_FILES:
    os.makedirs(FIG_DIR, exist_ok=True)
    fig.savefig(os.path.join(FIG_DIR, "01_240069_CP_raw_mean_range.png"), dpi=150)
plt.show()
plt.close(fig)


### 7. 114 마지막 분의 짧은 관측 표시하기

- 마지막 분도 저장함.
- 3초만 관측한 평균을 온전한 1분 기록처럼 모델에 넣지 않음.

In [ ]:
partial_start = pd.Timestamp("2024-11-04 20:57:00")
raw_end = df.loc[df["LOT_NO"].eq("240114") & df["MEAS_DT"].ge(partial_start)]
minute_end = minute.loc[minute["LOT_NO"].eq("240114") & minute["BIN_START"].ge(partial_start)]
display(minute_end[[
    "BIN_START", "OBS_START", "OBS_END", "ROW_COUNT", "OBSERVED_SPAN_SECONDS",
    "TC-Z3__COUNT", "EXCLUDED", "PARTIAL_MINUTE", "Z3__MODEL_ELIGIBLE", "Z3__MODEL_STATUS",
]].rename(columns={
    "BIN_START": "분 시작",
    "OBS_START": "첫 관측",
    "OBS_END": "마지막 관측",
    "ROW_COUNT": "원관측 수",
    "OBSERVED_SPAN_SECONDS": "관측 시간폭 초",
    "TC-Z3__COUNT": "온도 유효 수",
    "EXCLUDED": "기존 제외",
    "PARTIAL_MINUTE": "54초 미만",
    "Z3__MODEL_ELIGIBLE": "모델 입력 가능",
    "Z3__MODEL_STATUS": "보류 이유",
}))

fig, axis = plt.subplots(figsize=(12, 4))
axis.plot(raw_end["MEAS_DT"], raw_end["TC-Z3"], color="#396788", label="실제 TC-Z3 관측")
axis.scatter(minute_end["BIN_START"], minute_end["TC-Z3"], color="#d67b16", zorder=3, label="1분 평균, 분 시작에 표시")
for row in minute_end.loc[minute_end["PARTIAL_MINUTE"]].itertuples(index=False):
    axis.axvspan(row.BIN_START, row.BIN_END, color="#b7bdc5", alpha=0.3)
axis.set_title("114 마지막 분 | 회색 배경은 관측 시간폭 54초 미만")
axis.set_ylabel("TC-Z3 온도 (℃)")
axis.set_xlabel("2024년 실제 시각")
axis.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M:%S"))
axis.legend()
axis.grid(alpha=0.15)
fig.tight_layout()
if SAVE_FILES:
    fig.savefig(os.path.join(FIG_DIR, "02_240114_partial_minute.png"), dpi=150)
plt.show()
plt.close(fig)


### 8. v4와 원본 기록 연결표 저장하기

- `MEAS_DT` 컬럼은 분 시작 시각임.
- `OBS_START`와 `OBS_END` 컬럼은 실제 첫 관측과 마지막 관측 시각임.
- `AVAILABLE_AT`은 분이 끝나서 그 값을 사용할 수 있는 시각임.
  - `10:30:00~10:30:59`의 기록을 묶으면 `MEAS_DT=10:30:00`, `AVAILABLE_AT=10:31:00`임.
- `ROW_COUNT`는 이 행에 묶인 원본 기록 수이며 항상 60개인 것은 아님.
- v4 모델은 원본 기록에서 계산해 둔 60초 컬럼을 읽음.
  - 1분 자료에 15초 공백 기준이나 과거 10행 계산을 다시 적용하지 않음.
- 원본 기록 256개와 1분 기록 256개는 다른 양임.
  - 모델의 최소 학습 표본 옵션도 자료 단위에 맞춰 구분해야 함.
- 001을 시간순으로 나눌 때는 분할 경계를 걸친 분을 보류함.
  - `OBS_START`와 `OBS_END`로 실제 관측 범위를 확인함.
  - `AVAILABLE_AT`이 Train 끝을 넘는 분도 학습에서 뺌.
- 다른 CSV를 만들어도 이전 CSV의 생성 정보와 원본 연결표를 남김.
  - 두 파일은 `metadata/v4/`와 `review/v4/`에 자동 저장함.
  - 기존 `dataset.json`과 `raw_to_minute.csv`는 마지막 실행본임.

- 이전 생성 정보는 `history_...json`에도 남김.
  - 같은 CSV가 만들어져도 생성 근거가 달라지면 이전 기록을 덮지 않음.
  - 원본 연결표도 연결표 내용별 파일로 따로 보존함.


In [ ]:
row_map = df[["ROW_ID", "LOT_NO", "MEAS_DT", "SEGMENT", "MINUTE_ID"]].copy()
row_map = row_map.rename(columns={"ROW_ID": "SOURCE_ROW_ID", "MEAS_DT": "SOURCE_MEAS_DT"})
row_map["MINUTE_ROW_ID"] = row_map["MINUTE_ID"].map(minute["ROW_ID"])
row_map["BIN_START"] = row_map["MINUTE_ID"].map(minute["BIN_START"])
row_map["BIN_END"] = row_map["MINUTE_ID"].map(minute["BIN_END"])
row_map = row_map.drop(columns="MINUTE_ID")
output = minute.reset_index(drop=True)

dataset = {
    "dataset_version": "v4",
    "data_kind": "minute",
    "feature_profile": "raw60s_groupmean_1min",
    "master_file": OUTPUT_FILE,
    "source_file": INPUT_FILE,
    "source_sha256": input_hashes,
    "rows": len(output),
    "source_rows": len(df),
    "groups": groups,
    "grade_groups": GRADE_GROUPS,
    "input_columns_by_group": input_columns_by_group,
    "group_eligibility_suffix": "__MODEL_ELIGIBLE",
    "group_feature_count_suffix": "__FEATURE_COUNT",
    "label_suffix": "__LABEL",
    "aggregation": {
        "keys": aggregation_keys,
        "minute_seconds": 60,
        "observed_groups_only": True,
        "raw_sensor_fields": ["mean", "min", "max", "std_ddof1", "count"],
        "raw_mean": "원관측 산술평균, 시간 가중 평균 아님",
        "feature_mean": "같은 구역 입력이 모두 유효한 원행에서 기존 raw60초 특징의 산술평균",
        "feature_current_value": "원자료 sensor 평균과 sensor__VALUE는 입력 부족 원행 제외 여부 때문에 다를 수 있음",
        "grade_change_feature": "유효 구역 원행에서 log1p(강종변경 후 경과분)을 먼저 계산한 뒤 분 평균",
        "raw_window_seconds": 60,
        "raw_window_closed": "left",
        "raw_window_min_rows": MIN_PAST_ROWS,
        "raw_window_std_ddof": 1,
        "raw_feature_dtype_before_aggregation": "float32, 기존 IF98 특징과 동일",
        "raw_gap_seconds": MAX_GAP_SECONDS,
    },
    "model_eligibility": {
        "minimum_source_rows": MIN_MINUTE_ROWS,
        "minimum_observed_span_seconds": MIN_SPAN_SECONDS,
        "span_definition": "OBS_END-OBS_START+명목1초. 중간 결측을 채우거나 실제 관측초 수로 계산한 값은 아님.",
        "all_source_rows_must_have_group_features": True,
        "excluded_rows_eligible": False,
        "candidate_label_and_eligibility_are_separate": True,
        "minimum_values_are_domain_limits": False,
    },
    "label_policy": {
        "one": "제외되지 않은 분에 해당 구역 원라벨 1이 하나 이상 있음",
        "zero": "제외되지 않은 분의 해당 구역 원라벨이 모두 0",
        "missing": "기존 제외 또는 1이 없으면서 원라벨에 보류가 있음",
        "no_row_wide_or": True,
        "counts_and_ratio_preserved": True,
        "short_candidates_preserved": True,
        "first_last_times_do_not_fill_internal_gaps": True,
    },
    "time_policy": {
        "MEAS_DT": "BIN_START, 분 시작 표시용",
        "OBS_START": "실제 첫 원관측",
        "OBS_END": "실제 마지막 원관측",
        "AVAILABLE_AT": "BIN_END, 분이 끝난 뒤 사용 가능",
        "context": "집계 전에 원관측 시각으로 연결한 이벤트와 작업 맥락 보존",
        "split": "LOT 분할 유지. 내부 시간 분할은 OBS_START/OBS_END와 AVAILABLE_AT으로 경계 분 보류",
        "no_full_year_resampling": True,
    },
    "model_limits": [
        "분 모델은 제공된 특징을 직접 읽고 원관측 rolling을 재계산하지 않음",
        "최소 참조 256원행과 256분은 다르므로 분 모델 설정을 따로 기록",
        "raw min/max/std/count는 보존용이며 input_columns_by_group에 자동 포함하지 않음",
        "LOT 중앙60% 중앙값 기반 파생값은 미래 관측을 쓰므로 복사하지 않음",
        "정비 END_DT와 팀 제외는 사후 맥락이며 실시간 운전 명령이 아님",
        "구역 후보 비율의 단위가 초 행에서 분 행으로 바뀌므로 직접 같은 비율로 비교하지 않음",
    ],
    "row_map": "review/v4/raw_to_minute.csv",
    "issue": 100,
}

if SAVE_FILES:
    os.makedirs(META_DIR, exist_ok=True)
    os.makedirs(REVIEW_DIR, exist_ok=True)
    output.to_csv(OUTPUT_CSV, index=False, encoding=ENC, float_format="%.17g")
    with open(OUTPUT_CSV, "rb") as output_file:
        dataset["master_sha256"] = hashlib.sha256(output_file.read()).hexdigest()

    # 연결표 자체의 내용으로 이름을 정해 이전 연결표를 덮지 않음
    latest_map_path = os.path.join(REVIEW_DIR, "raw_to_minute.csv")
    row_map.to_csv(latest_map_path, index=False, encoding=ENC)
    with open(latest_map_path, "rb") as map_file:
        dataset["row_map_sha256"] = hashlib.sha256(map_file.read()).hexdigest()
    row_map_name = dataset["row_map_sha256"] + "_raw_to_minute.csv"
    row_map_path = os.path.join(REVIEW_DIR, row_map_name)
    if not os.path.isfile(row_map_path):
        row_map.to_csv(row_map_path, index=False, encoding=ENC)
    dataset["row_map"] = "review/v4/" + row_map_name

    # 기존 요약 이름은 가장 최근에 만든 파일로 유지함
    label_summary.to_csv(os.path.join(REVIEW_DIR, "label_summary.csv"), index=False, encoding=ENC)
    pd.DataFrame(group_quality).to_csv(os.path.join(REVIEW_DIR, "model_input_counts.csv"), index=False, encoding=ENC)
    # 같은 CSV라도 생성 근거가 달라지면 이전 설정 원문을 따로 보존함
    metadata_text = json.dumps(dataset, ensure_ascii=False, indent=2)
    metadata_sha256 = hashlib.sha256(
        json.dumps(dataset, ensure_ascii=False, sort_keys=True).encode("utf-8")
    ).hexdigest()
    for metadata_name in ["history_" + metadata_sha256 + ".json", dataset["master_sha256"] + ".json", "dataset.json"]:
        metadata_path = os.path.join(META_DIR, metadata_name)
        if metadata_name.startswith("history_") and os.path.isfile(metadata_path):
            continue  # 같은 설정 원문은 이미 저장한 파일을 사용함
        with open(metadata_path, "w", encoding="utf-8", newline="\n") as metadata_file:
            metadata_file.write(metadata_text)
    print("저장:", OUTPUT_CSV)
    print("원자료 연결:", row_map_path)
else:
    print("저장하지 않고 집계 결과만 확인함.")
display(output.head())
